# 第4章：图像滤波

## 编程实践：高斯滤波与双边滤波

---

## 一、图像滤波基础

### 1.1 什么是图像滤波？

图像滤波是利用**卷积核（Kernel/Filter）**对图像进行**局部运算**的过程。对于每个像素，取其周围邻域像素的值进行加权平均。

```
原始图像 (局部):         卷积核 (3×3):
  ┌───┬───┬───┐          ┌───┬───┬───┐
  │ 1 │ 2 │ 3 │          │ 1 │ 2 │ 1 │
  ├───┼───┼───┤          ├───┼───┼───┤
  │ 4 │ 5 │ 6 │    ×     │ 2 │ 4 │ 2 │  ÷ 16
  ├───┼───┼───┤          ├───┼───┼───┤
  │ 7 │ 8 │ 9 │          │ 1 │ 2 │ 1 │
  └───┴───┴───┘          └───┴───┴───┘

结果 = (1×1 + 2×2 + 3×1 + 4×2 + 5×4 + 6×2 + 7×1 + 8×2 + 9×1) / 16
     = (1 + 4 + 3 + 8 + 20 + 12 + 7 + 16 + 9) / 16 = 80/16 = 5.0
```

### 1.2 滤波的类型

| 类型 | 作用 | 特点 |
|------|------|------|
| 平滑滤波 | 减少噪声 | 模糊图像 |
| 高斯滤波 | 加权平滑 | 保留中心像素权重 |
| 中值滤波 | 去除椒盐噪声 | 非线性 |
| 双边滤波 | 保边去噪 | 同时考虑空间和值域 |
| 锐化滤波 | 增强边缘 | 高通滤波 |

### 1.3 高斯滤波原理

高斯滤波使用**高斯函数**作为卷积核的权重。距离中心越近的像素权重越大。

```
高斯函数: G(x,y) = (1/(2πσ²)) × exp(-(x²+y²)/(2σ²))

其中 σ (sigma) 控制滤波的平滑程度:
  - σ 越大 → 平滑程度越强
  - σ 越小 → 平滑程度越弱
```

### 1.4 双边滤波原理

双边滤波同时考虑**空间距离**和**像素值差异**：
- 空间权重：像素距离越近，权重越大（类似高斯）
- 值域权重：像素值越接近，权重越大
- 两者相乘得到最终权重
- 优点：可以在**去除噪声的同时保留边缘**


## 二、实现要求

### 实践1：高斯滤波
> 读取彩色图像，手写实现高斯滤波，保存结果。除 OpenCV 读写外，其余代码手写。

### 实践2：双边滤波
> 读取彩色图像，手写实现双边滤波，保存结果。除 OpenCV 读写外，其余代码手写。


In [ ]:
# 导入库
import cv2
import numpy as np
import math

print(f"OpenCV 版本: {cv2.__version__}")

In [ ]:
# 生成测试图像
import numpy as np

print("正在生成测试图像...")
h, w = 400, 600
img = np.zeros((h, w, 3), dtype=np.uint8)
for y in range(h):
    for x in range(w):
        img[y, x, 0] = int(100 + 80 * x / w)
        img[y, x, 1] = int(120 + 60 * y / h)
        img[y, x, 2] = int(80 + 100 * (x + y) / (w + h))
# 棋盘格纹理
cs = 40
for y in range(0, h, cs):
    for x in range(0, w, cs):
        if ((x // cs) + (y // cs)) % 2 == 0:
            img[y:y+cs, x:x+cs] = img[y:y+cs, x:x+cs] * 0.3
# 添加文字
cv2.putText(img, "Filter Test Image", (150, 220), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
# 添加高斯噪声
np.random.seed(42)
noise = np.random.normal(0, 25, img.shape).astype(np.float32)
img = np.clip(img.astype(np.float32) + noise, 0, 255).astype(np.uint8)
cv2.imwrite("filter_image.jpg", img)
print("测试图像已生成: filter_image.jpg")

In [ ]:
def gaussian_kernel(kernel_size, sigma):
    """
    手写生成高斯卷积核
    kernel_size: 卷积核大小 (奇数)
    sigma: 高斯标准差
    """
    # 确保 kernel_size 为奇数
    if kernel_size % 2 == 0:
        kernel_size += 1
    
    # 创建卷积核
    kernel = np.zeros((kernel_size, kernel_size), dtype=np.float64)
    center = kernel_size // 2
    
    # 计算每个位置的高斯权重
    for y in range(kernel_size):
        for x in range(kernel_size):
            # 相对于中心的偏移
            dx = x - center
            dy = y - center
            # 高斯函数
            kernel[y, x] = math.exp(-(dx**2 + dy**2) / (2 * sigma**2))
    
    # 归一化 (使权重和为 1)
    kernel = kernel / kernel.sum()
    
    return kernel


def gaussian_filter_manual(image, kernel_size=5, sigma=1.0):
    """
    手写高斯滤波 (对每个通道独立处理)
    """
    # 生成高斯核
    kernel = gaussian_kernel(kernel_size, sigma)
    
    height, width, channels = image.shape
    result = image.copy().astype('float64')
    
    center = kernel_size // 2
    
    # 对每个通道分别进行卷积
    for c in range(channels):
        for y in range(height):
            for x in range(width):
                # 对卷积核覆盖的区域进行加权求和
                weighted_sum = 0.0
                for ky in range(kernel_size):
                    for kx in range(kernel_size):
                        # 计算原图像中对应的坐标
                        img_y = y + ky - center
                        img_x = x + kx - center
                        
                        # 边界处理: 边界外像素用边界值
                        if 0 <= img_y < height and 0 <= img_x < width:
                            pixel_val = image[img_y, img_x, c]
                        elif img_y < 0:
                            pixel_val = image[0, img_x, c] if 0 <= img_x < width else 0
                        elif img_y >= height:
                            pixel_val = image[height-1, img_x, c] if 0 <= img_x < width else 0
                        elif img_x < 0:
                            pixel_val = image[img_y, 0, c] if 0 <= img_y < height else 0
                        elif img_x >= width:
                            pixel_val = image[img_y, width-1, c] if 0 <= img_y < height else 0
                        else:
                            pixel_val = 0
                        
                        weighted_sum += pixel_val * kernel[ky, kx]
                
                result[y, x, c] = weighted_sum
    
    return result.astype('uint8')

In [ ]:
# ==================== 实践1: 高斯滤波 ====================

# 读取带噪声的测试图像
img = cv2.imread("filter_image.jpg", cv2.IMREAD_COLOR)

if img is not None:
    h, w = img.shape[:2]
    print(f"读取图像成功! 尺寸: {w}x{h}")
    
    # 显示使用的高斯核
    kernel_size = 5
    sigma = 1.0
    kernel = gaussian_kernel(kernel_size, sigma)
    print(f"\n使用高斯核: {kernel_size}x{kernel_size}, sigma={sigma}")
    print(f"卷积核:\n{kernel}")
    
    # 执行高斯滤波
    print(f"\n正在进行高斯滤波... (逐像素处理可能较慢)")
    gaussian_result = gaussian_filter_manual(img, kernel_size=kernel_size, sigma=sigma)
    
    # 保存结果
    cv2.imwrite("gaussian_filtered.jpg", gaussian_result)
    print("滤波完成! 已保存: gaussian_filtered.jpg")
else:
    print("读取图像失败!")

In [ ]:
def bilateral_filter_manual(image, kernel_size=5, sigma_space=1.0, sigma_range=30.0):
    """
    手写双边滤波
    sigma_space: 空间域标准差 (控制空间权重)
    sigma_range: 值域标准差 (控制像素值差异权重)
    """
    height, width, channels = image.shape
    result = image.copy().astype('float64')
    
    center = kernel_size // 2
    
    # 归一化因子
    space_denom = 2 * sigma_space ** 2
    range_denom = 2 * sigma_range ** 2
    
    for c in range(channels):
        for y in range(height):
            for x in range(width):
                # 中心像素值
                center_val = float(image[y, x, c])
                
                weight_sum = 0.0
                weighted_val = 0.0
                
                for ky in range(kernel_size):
                    for kx in range(kernel_size):
                        # 计算邻域坐标
                        ny = y + ky - center
                        nx = x + kx - center
                        
                        # 边界处理 (使用镜像边界)
                        if ny < 0:
                            ny = -ny
                        elif ny >= height:
                            ny = 2 * (height - 1) - ny
                        if nx < 0:
                            nx = -nx
                        elif nx >= width:
                            nx = 2 * (width - 1) - nx
                        
                        # 获取邻域像素值
                        neighbor_val = float(image[ny, nx, c])
                        
                        # 空间权重 (基于距离)
                        spatial_weight = math.exp(-((ky-center)**2 + (kx-center)**2) / space_denom)
                        
                        # 值域权重 (基于像素值差异)
                        range_weight = math.exp(-((neighbor_val - center_val)**2) / range_denom)
                        
                        # 总权重
                        weight = spatial_weight * range_weight
                        
                        weight_sum += weight
                        weighted_val += neighbor_val * weight
                
                # 加权平均
                if weight_sum > 0:
                    result[y, x, c] = weighted_val / weight_sum
    
    return result.astype('uint8')

In [ ]:
# ==================== 实践2: 双边滤波 ====================

if img is not None:
    # 参数设置
    kernel_size = 5
    sigma_space = 1.0   # 空间域标准差
    sigma_range = 30.0  # 值域标准差 (像素值差异的容忍范围)
    
    print(f"正在进行双边滤波...")
    print(f"  核大小: {kernel_size}x{kernel_size}")
    print(f"  sigma_space: {sigma_space}")
    print(f"  sigma_range: {sigma_range}")
    
    bilateral_result = bilateral_filter_manual(
        img, 
        kernel_size=kernel_size, 
        sigma_space=sigma_space, 
        sigma_range=sigma_range
    )
    
    # 保存结果
    cv2.imwrite("bilateral_filtered.jpg", bilateral_result)
    print("\n滤波完成! 已保存: bilateral_filtered.jpg")
    
    # 对比显示
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[0].set_title('Original (with noise)')
    axes[0].axis('off')
    
    axes[1].imshow(cv2.cvtColor(gaussian_result, cv2.COLOR_BGR2RGB))
    axes[1].set_title('Gaussian Filter')
    axes[1].axis('off')
    
    axes[2].imshow(cv2.cvtColor(bilateral_result, cv2.COLOR_BGR2RGB))
    axes[2].set_title('Bilateral Filter')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print("\n对比说明:")
    print("  - 高斯滤波: 均匀模糊, 边缘也被模糊")
    print("  - 双边滤波: 保留边缘的同时去噪")

## 三、本章总结

### 高斯滤波 vs 双边滤波
| 特性 | 高斯滤波 | 双边滤波 |
|------|----------|----------|
| 权重计算 | 仅考虑空间距离 | 空间距离 + 像素值差异 |
| 边缘保持 | 差（模糊边缘） | 好（保留边缘） |
| 去噪能力 | 强 | 中等到强 |
| 计算量 | 较小 | 较大 |

### 关键参数
- **kernel_size**: 卷积核大小，越大滤波越强
- **sigma (σ)**: 控制权重分布的标准差
- **sigma_space**: 双边滤波的空间域参数
- **sigma_range**: 双边滤波的值域参数

### 注意事项
1. 卷积核大小必须为奇数
2. 边界处理方式影响结果（零填充/镜像/复制边界）
3. sigma 值建议取 kernel_size/6 左右
4. 彩色图像需要对每个通道独立处理
